In [1]:
import ollama
import pandas as pd
from yahoo_finance import MarketData
from news_embedder import NewsEmbedder

OLLAMA_HOST = "http://100.68.136.71:11434"

# Verifica se o modelo já está disponível antes de puxar
client = ollama.Client(host=OLLAMA_HOST)


# Etapa 2 — Engenharia de Features

Este notebook combina dados de mercado (Yahoo Finance) com embeddings de notícias (Ollama) em um dataset unificado para treinamento.

## 2.1 — Features de Mercado (`yahoo_finance.py`)

| Feature | Descrição |
|---------|-----------|
| `Close` | Preço de fechamento ajustado |
| `Volume` | Volume negociado |
| `return` | Retorno diário percentual (`pct_change()`) |
| `ma7`, `ma21` | Médias móveis de 7 e 21 dias |
| `std21` | Desvio padrão de 21 dias (proxy de volatilidade) |
| `lag_1`…`lag_5` | Valores defasados do fechamento |

**Total:** 11 features de preço por dia.

**[US-010]** O período padrão foi alterado de `period='5y'` para `period='max'` para buscar o máximo de histórico disponível no Yahoo Finance.

## 2.2 — Embeddings de Notícias (`news_embedder.py`)

- **Modelo:** `qwen3-embedding:4b` via Ollama → vetores de **1.024 dimensões** por artigo
- **Texto de entrada:** `title | excerpt | content` (truncado em 50.000 chars)
- **Agregação diária:** média ponderada por recência (`_weighted_mean`) — notícias mais recentes no dia recebem peso maior
- **Cache:** `embeddings_cache.npz` — artigos já embedados são reutilizados sem nova chamada ao modelo

## 2.3 — Merge Temporal (`merge_with_prices()`)

1. **Left join** por data: combina features de preço com embeddings diários
2. **Forward-fill:** propaga último embedding para dias sem notícias
3. **Drop NaN:** remove linhas com valores ausentes

## Dataset Atual

- **Arquivo:** `dataset_full.csv`
- **Dimensões:** 1.227 dias × 1.035 features (11 preço + 1.024 embeddings)
- **Período:** 2021-04-28 a 2026-03-26 (~5 anos)
- **Artigos:** 2.572 embedados → 1.115 dias únicos com notícias

## Expansão US-010

Para reconstruir o dataset com período máximo:
```python
md = MarketData("ITUB4.SA")
X_price = md.features(period="max", lags=5)  # agora é o default
```
O `embeddings_cache.npz` é reaproveitado automaticamente pelo `NewsEmbedder`.

In [2]:
client.list().models

00:10:23 [INFO] HTTP Request: GET http://100.68.136.71:11434/api/tags "HTTP/1.1 200 OK"


[Model(model='qwen3-embedding:4b', modified_at=datetime.datetime(2026, 3, 16, 11, 3, 45, 563525, tzinfo=TzInfo(-10800)), digest='df5bd2e3c74cd8d069d21dc038f1b359fcdc9458fce1c99bd43c9eb1518ff907', size=2496704041, details=ModelDetails(parent_model='', format='gguf', family='qwen3', families=['qwen3'], parameter_size='4.0B', quantization_level='Q4_K_M')),
 Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 3, 2, 20, 16, 31, 728797, tzinfo=TzInfo(-10800)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M')),
 Model(model='lfm2.5-thinking:latest', modified_at=datetime.datetime(2026, 3, 2, 20, 12, 22, 506453, tzinfo=TzInfo(-10800)), digest='95bd9d45385f33bfe96d8b3651c8569e152f21f5bdb7c19894ffde650e9cf140', size=731163903, details=ModelDetails(parent_model='', format='gguf', family='lfm2', familie

In [3]:
embedding_model = "qwen3-embedding:0.6b"

In [4]:
modelos = [m.model for m in client.list().models]
if embedding_model not in modelos:
    print(f"Baixando {embedding_model}...")
    client.pull(embedding_model)

00:10:23 [INFO] HTTP Request: GET http://100.68.136.71:11434/api/tags "HTTP/1.1 200 OK"


In [5]:

# Notícias
df_news = pd.read_json("../1.news/itub4_noticias.json", convert_dates=False)
articles  = df_news.to_dict(orient="records")

# Preços — período estendido para 5 anos (alinhado com dados de notícias desde 2021)
md      = MarketData("ITUB4.SA")
X_price = md.features(period="5y", lags=5)
y       = md.target(period="5y", horizon=5)

# Embedder
ne = NewsEmbedder(
    model=embedding_model,
    fields=["title", "excerpt", "content"],
    ollama_host=OLLAMA_HOST,
    cache_path="embeddings_cache.npz",
    summarizer_model='lfm2.5-thinking'
)


00:10:24 [INFO] Cache carregado: 2572 embeddings de 'embeddings_cache.npz'


In [6]:
df_news

,id,date,modified,title,link,excerpt,content,author_id,author_name,hat,categories,tags,keywords
0,3252277,2026-03-16T13:08:54,2026-03-16T13:09:01,Cosan (CSAN3): o que esperar da holding após r...,https://www.infomoney.com.br/mercados/cosan-cs...,Cosan vem passando por um processo de reestrut...,Após a Cosan (CSAN3) divulgar seus resultados ...,54205,Felipe Moreira,De olho em CSAN3,[1],"[332, 2691, 629, 210752, 381]","[Ações, Cosan, Goldman Sachs, Recomendação neu..."
1,3251628,2026-03-16T10:56:56,2026-03-16T10:57:01,"Com maior peso de estatais na AL, Brasil abre ...",https://www.infomoney.com.br/mercados/com-maio...,Banco atualizou sua cesta de ações de estatais...,"De olho na proximidade do pleito de outubro, o...",25,Lara Rizério,Cenário para eleições,[1],"[288694, 1172, 1414, 1440, 1880, 12, 2517]","[Axia, Bradesco BBI, Embraer, Estatais, MSCI, ..."
2,3250267,2026-03-16T06:04:39,2026-03-16T06:06:01,"Prévia do PIB, indústria nos EUA, conflito no ...",https://www.infomoney.com.br/mercados/previa-p...,InfoMoney reúne as principais informações que ...,A sessão desta segunda-feira (16) atualiza mai...,620517,"Erick Souza, Felipe Moreira",O que vai mexer com a Bolsa,[1],"[986, 382, 1383, 283800, 539, 192609, 194340, ...","[5 Assuntos, Banco Central, Donald Trump, Elei..."
3,3250055,2026-03-13T17:51:27,2026-03-13T18:19:20,Ibovespa fecha abaixo de 178 mil pontos sem al...,https://www.infomoney.com.br/mercados/ibovespa...,O volume financeiro nesta sexta-feira somou R$...,"SÃO PAULO, 13 Mar (Reuters) – O Ibovespa fecho...",43,Reuters,Mercado doméstico,[1],"[653, 1649, 2314]","[Ibovespa, Irã, Trader]"
4,3250081,2026-03-13T18:13:47,2026-03-13T18:13:49,"Previ recebeu R$ 4,4 bi em dividendos e JCP e ...",https://www.infomoney.com.br/onde-investir/pre...,Dividendos e juros sobre capital próprio ajuda...,"A Previ recebeu R$ 4,4 bilhões em dividendos e...",44,Estadão Conteúdo,Dividendos e JCP,[2476],"[527, 1695, 814]","[Dividendos, JCP, Previ]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2567,1626362,2021-04-20T08:20:50,2021-04-20T09:19:21,Dados de produção da Vale e de vendas do Carre...,https://www.infomoney.com.br/mercados/dados-de...,Confira os destaques do noticiário corporativo...,SÃO PAULO – O noticiário corporativo desta ter...,293,Equipe InfoMoney,Radar InfoMoney,[1],"[332, 1227, 2803, 2068, 2517]","[Ações, Carrefour, Linx, Radar InfoMoney, Vale]"
2568,1625265,2021-04-16T12:23:51,2021-04-16T12:25:03,B3 mantém Locaweb (LWSA3) e inclui units do Ba...,https://www.infomoney.com.br/mercados/b3-mante...,"Com isso, caso confirmada na próxima prévia, a...",A B3 divulgou nesta sexta-feira (16) a segunda...,293,Equipe InfoMoney,Nova carteira,[1],"[332, 2840, 653, 1742]","[Ações, Banco Inter, Ibovespa, Locaweb]"
2569,1624516,2021-04-15T17:44:23,2021-04-15T17:45:41,"Ações de Vale e frigoríficos voltam a subir, P...",https://www.infomoney.com.br/mercados/acoes-de...,Confira os destaques da B3 na sessão desta qui...,SÃO PAULO – O grande destaque do mercado nesta...,25,Lara Rizério,Destaques da Bolsa,[1],"[332, 2782, 2445, 2614, 2495, 2712, 2490]","[Ações, Arezzo, Bradesco, Cia Hering, Itaú Uni..."
2570,1623847,2021-04-14T17:40:46,2021-04-14T18:01:26,"Ações de Vale, siderúrgicas, Petrobras e frigo...",https://www.infomoney.com.br/mercados/acoes-da...,Confira os destaques da B3 na sessão desta qua...,SÃO PAULO – O destaque do Ibovespa na sessão d...,25,Lara Rizério,Destaques da Bolsa,[1],"[332, 2550, 2687, 2827, 2713, 2517]","[Ações, Americanas, B2W Digital, CVC Brasil, J..."


In [7]:
# Dataset final
X_full = ne.merge_with_prices(X_price, articles)
df     = X_full.join(y).dropna()
X      = df.drop(columns=["target"])
y      = df["target"]

print(f"Shape final: {X.shape}")  # (dias, price_features + 768 emb dims)

00:10:24 [INFO] Iniciando embedding de 2572 artigos com 'qwen3-embedding:0.6b'


00:10:24 [INFO]   [   1/2572] 2026-03-16 | cache |   0.0s | ETA 0.0s | chars=6562


00:10:24 [INFO]   [   2/2572] 2026-03-16 | cache |   0.0s | ETA 0.0s | chars=5344


00:10:24 [INFO]   [   3/2572] 2026-03-16 | cache |   0.0s | ETA 0.0s | chars=5748


00:10:24 [INFO]   [   4/2572] 2026-03-13 | cache |   0.0s | ETA 0.0s | chars=5408


...

00:10:26 [INFO]   Tempo embed       : 0.0s


00:10:26 [INFO]   Tempo total       : 1.0s


00:10:26 [INFO] ───────────────────────────────────────────────────────


00:10:26 [INFO] Embeddings prontos: 1115 dias únicos


Shape final: (1222, 1035)


In [8]:
X_full

,Close,Volume,return,ma7,ma21,std21,lag_1,lag_2,lag_3,lag_4,...,emb_1014,emb_1015,emb_1016,emb_1017,emb_1018,emb_1019,emb_1020,emb_1021,emb_1022,emb_1023
Date,,,,,,,,,,,,,,,,,,,,,
2021-04-28,18.178738,42502908,0.043520,17.704371,17.666914,0.311431,17.420595,17.690685,17.703552,17.619949,...,0.017550,0.006109,-0.011335,-0.020358,-0.014249,0.031750,-0.026508,-0.000099,-0.003484,-0.027420
2021-04-29,17.605213,37968076,-0.031549,17.690322,17.646463,0.299963,18.178738,17.420595,17.690685,17.703552,...,0.018647,0.005884,0.003188,-0.004949,-0.003468,0.024642,-0.012430,-0.012247,-0.028131,-0.004347
2021-04-30,17.740538,48196913,0.007687,17.708467,17.618377,0.257308,17.605213,18.178738,17.420595,17.690685,...,0.018647,0.005884,0.003188,-0.004949,-0.003468,0.024642,-0.012430,-0.012247,-0.028131,-0.004347
2021-05-03,17.988770,46978032,0.013992,17.761156,17.618945,0.258150,17.740538,17.605213,18.178738,17.420595,...,-0.015276,0.029940,-0.032191,-0.033473,-0.001386,0.011107,0.011976,-0.000753,-0.012030,-0.012641
2021-05-04,17.221508,89509379,-0.042652,17.692292,17.609158,0.269438,17.988770,17.740538,17.605213,18.178738,...,0.011418,0.021940,-0.029372,-0.020259,-0.030225,0.010395,-0.003080,0.008929,-0.004219,-0.012892
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-20,41.549999,37660900,-0.016128,42.159321,44.259680,2.288494,42.231121,41.933578,42.360054,42.647678,...,0.012914,0.040775,0.002258,-0.018842,-0.031168,0.001763,-0.006106,0.002091,0.009380,0.001340
2026-03-23,42.779999,27535500,0.029603,42.222147,43.972648,2.055716,41.549999,42.231121,41.933578,42.360054,...,0.012914,0.040775,0.002258,-0.018842,-0.031168,0.001763,-0.006106,0.002091,0.009380,0.001340
2026-03-24,42.540001,18609000,-0.005610,42.291776,43.758223,1.951662,42.779999,41.549999,42.231121,41.933578,...,0.012914,0.040775,0.002258,-0.018842,-0.031168,0.001763,-0.006106,0.002091,0.009380,0.001340


In [9]:
X_full.to_csv("dataset_full.csv", index=True)